# Constraint Diagram
- We need to rigorously update this: Examine the weights, everything is a fraction but the weights (MTOW) changes.
- Develop accurate equations for a propeller.

---

#### Future Work: Sensitivity Analysis on:
- $\frac{P}{W}$ depending on Velocities at different conditions, AR, $eta_p$ at different conditions, $C_{L,Max}$, $C_{D,0}$


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from ambiance import Atmosphere
import pint
ureg = pint.UnitRegistry()

Fundamental Constants:

---

In [2]:
MTOW=50000*ureg.pound_force #[lbf] note: PLACEHOLDER
eta_pcruise=0.9
C_D0=0.02 # Taken from The annual aircraft design book (blue), page 6-7. 
AR=10 # competitively assessed
e=0.85 # competitively assessed (at cruise)
k=1/(np.pi*e*AR)
M_cruise=0.55 #cruise mach, competitively assessed 
cruise_alt=20000*ureg.feet #cruise altitude [input feet], competitively assessed
landing_distance=(0.6*5000*ureg.feet).magnitude # [ft] landing and takeoff distance at sea level, competitively estimated
#take_off_alt=(0*ureg.feet).magnitude #[ft] altitude of takeoff


$$ W_l= X * MTOW $$

This was derived with the following: 
$ W_L=W_{TO}\left[\frac{W_{pl}}{W_{TO}}+EWF+  (\approx{0})\right] $ From sizing excel sheet we have an estimate for $W_{TO}, EWF, \frac{W_{pl}}{W_{TO}}$

In [3]:
EWF=0.6 # from carpet plot spreadsheet
W_p=16250*ureg.pound_force # spreadsheet, 200 lbs * 65 people on plane
Fuel_fraction=1-EWF-W_p/MTOW # fuel fractionFuel_fraction = (1 - EWF - W_p / MTOW).to("").magnitude
if not (0 < Fuel_fraction < 1):
    raise ValueError("MTOW, EWF, and payload are inconsistent.")
fuel_remaining_fraction_cruise = 0.62 # these values were estimated with a range breakdown
fuel_remaining_fraction_landing = 0.24

beta_landing=(W_p/MTOW)+EWF+(Fuel_fraction*fuel_remaining_fraction_landing) # W_l/MTOW
beta_cruise=(Fuel_fraction*fuel_remaining_fraction_cruise)+(W_p/MTOW)+EWF # W_cruise/MTOW

print("Landing weight fraction: ", beta_landing)
print("Cruise weight fraction: ", beta_cruise)
print("Fuel Fraction: ", Fuel_fraction*100,"%")

Landing weight fraction:  0.9430000000000001 dimensionless
Cruise weight fraction:  0.9715 dimensionless
Fuel Fraction:  7.500000000000001 dimensionless %


Cruise Velocity Constraint: We need to be able to maintain a cruise speed


In [4]:
# cruise velocity constraint

sealevel = Atmosphere(0) # gives SI 
# NOTE: THIS CHANGES WITH ALTITUDE, current: 20000 ft
a_cruisealt = (Atmosphere(cruise_alt.to("meters").magnitude).speed_of_sound)*ureg("m/s").to("ft/s").magnitude # ISA to find speed of sound, convert to (ft/s)
rho_cruisealt=Atmosphere(cruise_alt.to("meters").magnitude).density*ureg("kg/m^3").to("slug/ft^3").magnitude # ISA to find density at cruise altitude (slug/ft^3)
rho_sl=sealevel.density*ureg("kg/m^3").to("slug/ft^3").magnitude # sea-level slug/ft^3
V_cruise=M_cruise*a_cruisealt #ft/sec
W_S=np.linspace(1, 200, 100) # linspace for wingload
q_cruise=0.5*rho_cruisealt*V_cruise**2 # dynamic pressure at cruise

alpha_T_cruise=1 # NOTE: Input engine model later
alpha_P_cruise=1 # cruise power available/sea-level power estimate
T_W_cruise=(q_cruise*C_D0*(1/W_S)+(k/q_cruise)*W_S)/alpha_T_cruise # above line = valid

P_W_cruise =( 
    (V_cruise / 
    (550 * eta_pcruise * alpha_P_cruise))*
    (q_cruise*C_D0/W_S +(k*(W_S/q_cruise)*beta_cruise**2)) # the 550 is to get horsepower
)



Rate of Climb Constraint: Directly from the FARs.
- OEI: One engine inoperative


In [5]:
# Certification basis: 14 CFR 25.121(b), if the program is pursuing Part 25.
# For a twin, required gross climb gradient is 2.4% at V2, one engine inoperative,
# landing gear retracted, and takeoff flap configuration.

N_engines = 2
gamma_oei_2nd = 0.024  # source: 14 CFR 25.121(b), twin-engine requirement.

beta_oei_2nd = 1.0
# Preliminary approximation. Takeoff-segment mission weight when available.

C_L_max_TO = 1.7 # Roskam pp. 91 and 377
                 # Replace after high-lift sizing or competitive-data validation.

V2_factor = 1.13
# [-]; source: 14 CFR 25.107(b)(1), minimum V2 factor for a two-engine turboprop.
# This is a screening lower bound only; final V2 must also satisfy VMCG, VMCA,
# rotation, maneuvering, and takeoff-path requirements.

delta_C_D0_TO = None
# source required: team's takeoff-configuration drag buildup.
# Includes flap drag in the gear-retracted takeoff-climb configuration.

delta_C_D0_OEI = None
# source required: OEI drag assessment.
# Includes feathered/windmilling propeller, asymmetric-rudder trim, and nacelle effects.

e_TO = e
# source required: takeoff-configuration drag-polar estimate.
# ASSUMING e_cruise (oslwald) FOR NOW.

eta_p_V2 = None
# source required: propeller map or manufacturer data at V2 and takeoff RPM.

alpha_P_takeoff = None
# source required: engine deck at the specified airport pressure altitude,


if None in (delta_C_D0_TO, delta_C_D0_OEI, e_TO, eta_p_V2, alpha_P_takeoff):
    raise ValueError("Document takeoff drag, OEI drag, efficiency, and engine-deck inputs.")

rho_takeoff = rho_sl # [slug/ft^3]

C_D0_oei_TO = C_D0 + delta_C_D0_TO + delta_C_D0_OEI
k_TO = 1.0 / (np.pi * e_TO * AR)

# Reference stall speed and screening V2, both in ft/s.
V_SR_TO = np.sqrt(2 * beta_oei_2nd * W_S / (rho_takeoff * C_L_max_TO))
V2 = V2_factor * V_SR_TO
q_V2 = 0.5 * rho_takeoff * V2**2

D_W_oei_TO = (
    q_V2 * C_D0_oei_TO / W_S
    + k_TO * beta_oei_2nd**2 * W_S / q_V2
)

# P_SL/W_TO for total installed sea-level-rated shaft power.
# N/(N-1) accounts for one unavailable engine.
P_W_oei_2nd = (
    (N_engines / (N_engines - 1))
    * V2
    * (D_W_oei_TO + gamma_oei_2nd)
    / (550 * eta_p_V2 * alpha_P_takeoff)
)

ValueError: Document takeoff drag, OEI drag, efficiency, and engine-deck inputs.

Service Ceiling Constraint: Must be able to operate at a certain service ceiling.
- Status: Needs oversight, value estimations, and equation verification!

In [ ]:
# Definition: 100 ft/min maximum ROC, clean configuration, maximum weight,
# and maximum continuous power (FAA Airplane Flying Handbook).

h_service_ceiling = 25000*ureg.feet # [ft]; competitive requirement.
ROC_service = 100.0 / 60.0 # [ft/s]; source: FAA service-ceiling definition of 100 ft/min.
beta_ceiling = None
# mission fuel accounting at the ceiling condition.
# Use W_at_ceiling / W_TO.
alpha_P_MCP = None
# source required: engine deck, maximum-continuous shaft power at ceiling altitude.
eta_p_ceiling = None # source required: propeller map at the ceiling climb condition.

if None in (h_service_ceiling, beta_ceiling, alpha_P_MCP, eta_p_ceiling):
    raise ValueError("Document the ceiling altitude, mission weight, engine deck, and propeller efficiency.")

atm_ceiling = Atmosphere((h_service_ceiling * ureg.feet).to("meters").magnitude)
rho_ceiling = atm_ceiling.density * ureg("kg/m^3").to("slug/ft^3").magnitude #density at cruise

V_ceiling = np.linspace(100.0, 500.0, 401)  # [ft/s]; numerical search range, verify after convergence.
q_ceiling = 0.5 * rho_ceiling * V_ceiling[:, None]**2
D_W_ceiling = (
    q_ceiling * C_D0 / W_S[None, :]
    + k * beta_ceiling**2 * W_S[None, :] / q_ceiling
)
P_W_ceiling_by_speed = (
    ROC_service + V_ceiling[:, None] * D_W_ceiling
) / (550 * eta_p_ceiling * alpha_P_MCP)
# For each W/S, choose the speed requiring the least installed power.
P_W_service_ceiling = np.min(P_W_ceiling_by_speed, axis=0)

Landing Constraint: Must be able to land a certain distance at stall speed.

In [ ]:
# landing constraint
V_approach=np.sqrt(landing_distance/0.3) #roskam page 111 (NOTE: verify this works for part 25)

V_stall = V_approach/1.3 # page 111 roskam , in knots, need to convert to ft/sec
V_stall = V_stall*1.68781 # convert knots to ft/sec
C_L_max_L = 2.15 # page 91 roskam (estimated, competitive)
W_S_landing = 0.5 * rho_sl * V_stall**2 * C_L_max_L / beta_landing # roskam equation 3.1 , then 
                                                                    #normalize by weight fraction to get in terms of takeoff w/s
print(f"W_S_landing: {W_S_landing}")

W_S_landing: [76.12292324]


In [6]:
# AIAA-style constraint diagram
# The 7.0-in width is suitable for a two-column AIAA figure.
from IPython.display import Math, display
plt.rcParams.update({
    'font.family': 'times new roman',
    'font.serif': ['STIX Two Text', 'Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'font.size': 12,
    'axes.labelsize': 10,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.linewidth': 0.8,
})
# Required installed sea-level shaft-power loading at each W/S.
P_W_required = np.maximum.reduce([
    P_W_cruise,
    P_W_oei_2nd,
    P_W_service_ceiling,
])

# The landing constraint is an upper limit on takeoff wing loading.
feasible = W_S <= W_S_landing
if not np.any(feasible):
    raise ValueError("Landing constraint lies outside the W/S grid.")

# Lowest installed-power design that satisfies every plotted constraint.
P_W_feasible = np.where(feasible, P_W_required, np.inf)
i_design = np.argmin(P_W_feasible)

W_S_design = W_S[i_design]
P_W_design = P_W_required[i_design]

fig, ax = plt.subplots(
    figsize=(7.0, 4.5),
    dpi=200,
    constrained_layout=True,
)

cruise_line, = ax.plot(
    W_S, P_W_cruise, color="black", lw=1.6,
    label=rf"Cruise, $M={M_cruise:.2f}$", zorder=3,
)

oei_line, = ax.plot(
    W_S, P_W_oei_2nd, color="tab:blue", lw=1.6,
    label=r"Second-segment OEI climb", zorder=3,
)

ceiling_line, = ax.plot(
    W_S, P_W_service_ceiling, color="tab:purple", lw=1.6,
    label=rf"Service ceiling, ${h_service_ceiling.magnitude:.0f}\,\mathrm{{ft}}$",
    zorder=3,
)

landing_line = ax.axvline(
    W_S_landing, color="tab:red", lw=1.6,
    label=rf"Landing limit, $W_{{TO}}/S={W_S_landing:.1f}\,\mathrm{{lbf/ft^2}}$",
    zorder=3,
)

for line, color in [
    (cruise_line, "black"),
    (oei_line, "tab:blue"),
    (ceiling_line, "tab:purple"),
    (landing_line, "tab:red"),
]:
    line.set_path_effects([
        pe.withTickedStroke(
            spacing=10, angle=585, length=0.6, foreground=color
        )
    ])

ax.scatter(
    W_S_design, P_W_design, s=75, marker="o",
    facecolor="green", edgecolor="black", linewidth=1.1, zorder=6,
    label="Selected design point",
)

x_visible = (W_S >= 20) & (W_S <= 1.05 * W_S_landing)
y_top = 1.12 * np.max(P_W_required[x_visible])

ax.set_xlim(20, 1.05 * W_S_landing)
ax.set_ylim(0, y_top)
ax.set_xlabel(r"Wing loading, $W_{TO}/S$ [$\mathrm{lbf/ft^2}$]")
ax.set_ylabel(r"Installed shaft-power loading, $P_{SL}/W_{TO}$ [$\mathrm{shp/lbf}$]")
ax.tick_params(which="both", direction="in", top=True, right=True,
               width=0.8, length=4)
ax.grid(True, which="major", color="0.87", linewidth=0.6, zorder=0)

legend = ax.legend(
    loc="best", frameon=True, fancybox=False, framealpha=1.0,
    facecolor="white", edgecolor="black", borderpad=0.3, handlelength=3.0,
)
legend.get_frame().set_linewidth(0.7)

fig.savefig(
    "sizing_constraint_diagram_AIAA.png",
    dpi=300, facecolor="white", bbox_inches="tight",
)

W_S_design_qty = W_S_design * ureg.pound_force / ureg.feet**2
S = (MTOW / W_S_design_qty).to("ft^2")
P_shaft = (P_W_design * ureg.horsepower / ureg.pound_force * MTOW).to("horsepower")
b = np.sqrt(S.magnitude * AR)
c_bar = b / AR

print(f"Selected design point: W/S = {W_S_design:.1f} lbf/ft²")
print(f"Required installed shaft power: {P_shaft.magnitude:.0f} shp")
print(f"Wing area: {S.magnitude:.1f} ft²")
print(f"Span: {b:.1f} ft, mean chord: {c_bar:.1f} ft")

NameError: name 'P_W_oei_2nd' is not defined

# Assumptions:
- Roskam $s_l=0.3V_{approach}^2$ is used as an early sizing approximation
- `beta_landing` includes all reserve fuel, basically all fuel right at landing.